# Volume breakout: backtest

Trigger: close above the prior 30-day high, on at least 2.5x median volume, with high delivery %, on a stock that passes the tradability filter.

Tune on **2018–2022 only**. Look at 2023+ once, at the end. Changing a parameter after seeing 2023+ results turns them into in-sample results.

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.simplefilter('ignore')
import pandas as pd
from nsesig.config import Settings, VolumeBreakoutConfig
from nsesig.data import load_cache
from nsesig.synthetic import make_panel
from nsesig.backtest import run_volume_breakout

USE_SYNTHETIC = True  # set False once data/bhav is populated (scripts/fetch_bhavcopy.py)
panel = make_panel() if USE_SYNTHETIC else load_cache('../data/bhav')
panel.shape

## Optional inputs

Without these, the matching filter is skipped with a warning:
- **ASM/GSM history**: `universe.attach_surveillance(panel, lists)`
- **Shares outstanding** (for market cap): `universe.attach_mcap(panel, shares)`
- **Sector map** (for the sector cap): pass `sectors={symbol: sector}` to the runner

In [ ]:
settings = Settings(breakout=VolumeBreakoutConfig(volume_mult=2.5, high_lookback=30))
r = run_volume_breakout(panel, settings)
print('target (in-sample median move):', round(r.target_pct, 4))
r.verdict

In [ ]:
pd.DataFrame({p.name: {**p.events, **{k: v for k, v in p.trades.items() if k != 'exit_reasons'},
                        **{'pre_tax_'+k: v for k, v in p.pre_tax.items()}, **{'post_tax_'+k: v for k, v in p.post_tax.items()}}
              for p in (r.in_sample, r.out_of_sample)})

In [ ]:
ax = r.in_sample.equity_post_tax.plot(label='in-sample', figsize=(10, 4))
r.out_of_sample.equity_post_tax.plot(ax=ax, label='out-of-sample')
ax.set_title('Post-tax realised equity'); ax.legend();

## Sensitivity (in-sample only)

If the edge exists only at one exact parameter value, it is probably noise.

In [ ]:
rows = []
for vm in (2.0, 2.5, 3.0):
    for lb in (20, 30, 40):
        s = Settings(breakout=VolumeBreakoutConfig(volume_mult=vm, high_lookback=lb))
        res = run_volume_breakout(panel, s)
        rows.append({'volume_mult': vm, 'lookback': lb, **{k: res.in_sample.trades.get(k) for k in ('trades', 'hit_rate', 'expectancy', 'profit_factor')}})
pd.DataFrame(rows)